# 경구약제 118종 분류기 학습

데이터 구성 모듈에서 `train_loader`, `val_loader`, `class_map`, `data_summary`를 받은 뒤 학습·재개·평가·artifact 저장을 실행한다. 모델, epoch loop, 지표, checkpoint 함수는 프로젝트 모듈을 사용한다.

In [ ]:
from importlib import import_module
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn

from classifier import (
    TrainConfig,
    evaluate_with_predictions,
    load_inference_model,
    run_training,
    save_validation_artifacts,
    select_device,
    validate_batch,
    validate_class_map,
    validate_data_summary,
)

In [ ]:
run_name = "resnet50_baseline"
data_factory_path = ""  # 예: "data.classifier_data:build_classifier_inputs"
output_root = project_root / "outputs"

config = TrainConfig(
    run_name=run_name,
    output_dir=output_root,
    class_map_path=project_root / "data" / "class_map.json",
    data_summary_path=project_root / "data" / "data_summary.json",
)
device = select_device()
print(f"device={device}")

## 데이터 연결

`data_factory_path`가 가리키는 함수는 `factory(config=config, device=device)` 호출을 지원하고 `(train_loader, val_loader, class_map, data_summary)`를 반환해야 한다. 실제 데이터 모듈 경로가 확정되면 설정 셀의 문자열만 변경한다.

In [ ]:
if not data_factory_path:
    raise RuntimeError("data_factory_path를 데이터 구성 모듈의 함수로 설정하세요")

module_name, function_name = data_factory_path.split(":", maxsplit=1)
data_factory = getattr(import_module(module_name), function_name)
train_loader, val_loader, class_map, data_summary = data_factory(
    config=config,
    device=device,
)

In [ ]:
validate_class_map(class_map)
validate_data_summary(data_summary, class_map)
validate_batch(next(iter(train_loader)), class_map)
validate_batch(next(iter(val_loader)), class_map)

# 입력 검사에서 sampler 상태가 바뀌지 않도록 학습용 loader를 새로 만든다.
train_loader, val_loader, class_map, data_summary = data_factory(
    config=config,
    device=device,
)
print("입력 검증 완료")

## 학습 또는 재개

재개할 때는 `config.resume_from`을 해당 실행의 `last.pt`로 설정한 뒤 같은 셀을 실행한다.

In [ ]:
# config.resume_from = output_root / run_name / "last.pt"
training_result = run_training(
    config,
    train_loader,
    val_loader,
    class_map,
    data_summary,
    device=device,
)
training_result

In [ ]:
metrics = pd.read_csv(training_result.run_dir / "metrics.csv")
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
metrics.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0, 0])
metrics.plot(x="epoch", y=["train_top1_accuracy", "val_top1_accuracy"], ax=axes[0, 1])
metrics.plot(x="epoch", y=["train_top5_accuracy", "val_top5_accuracy"], ax=axes[1, 0])
metrics.plot(x="epoch", y="val_macro_f1", ax=axes[1, 1])
fig.tight_layout()

## Best checkpoint 평가와 오류 저장

In [ ]:
best_model, device = load_inference_model(
    training_result.best_checkpoint_path,
    class_map,
    device=device,
)
artifacts = evaluate_with_predictions(
    best_model,
    val_loader,
    class_map,
    nn.CrossEntropyLoss(),
    device,
)
save_validation_artifacts(artifacts, training_result.run_dir)
artifacts.metrics.to_dict()

In [ ]:
class_metrics = pd.DataFrame(
    [metric.to_dict() for metric in artifacts.metrics.per_class]
)
display(class_metrics.sort_values(["support", "recall"], ascending=[True, True]).head(20))

error_samples = pd.DataFrame(
    [sample.to_dict() for sample in artifacts.error_samples]
)
display(error_samples.head(20))